In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload


In [2]:
import sys
from pathlib import Path
from src.phaseI.main import run_phase1
from src.phaseII.main import run_phase2

from demo.core.data import (
    SplitConfig,
    prepare_cmapss_split,
)


config = SplitConfig(
    data_path="demo/data/train_FD001.txt",

    train_ratio=0.70,

    elbow_cycle=130,
    elbow_tolerance=5,

    extended_probability=0.30,
    extended_max_extra=50,

    min_cycles_before_failure=20,

    test_fraction_min=0.10,
    test_fraction_max=0.30,

    random_seed=42,
)
data = prepare_cmapss_split(config)

from src.preprocess_data.realtime import (
    simulate_realtime,
    RealtimeConfig,
)

realtime = simulate_realtime(
    engine_data=data["engine_data"],
    test_data=data["test_data"],
    observation_points=data["test_observation_points"],
    config=RealtimeConfig(
        update_probability=0.8,
        max_new_cycles=1,
        random_seed=42,
    ),
)


In [3]:
BIN_STRIDE = 10

model, stats, normalized_data, latent_data = run_phase1(
    train_data=data["train_data"],
    n_sensors=len(data["sensors"]),
    window_len=30,
    latent_dim=16,
    bin_stride=BIN_STRIDE,
)


epoch   0 | recon_loss=0.641268
epoch   1 | recon_loss=0.459071
epoch   2 | recon_loss=0.455278
epoch   3 | recon_loss=0.452914
epoch   4 | recon_loss=0.451605
epoch   5 | recon_loss=0.449937
epoch   6 | recon_loss=0.448773
epoch   7 | recon_loss=0.447074
epoch   8 | recon_loss=0.444165
epoch   9 | recon_loss=0.442259
epoch  10 | recon_loss=0.440754
epoch  11 | recon_loss=0.438643
epoch  12 | recon_loss=0.436481
epoch  13 | recon_loss=0.434676
epoch  14 | recon_loss=0.432115
epoch  15 | recon_loss=0.428120
epoch  16 | recon_loss=0.423622
epoch  17 | recon_loss=0.420929
epoch  18 | recon_loss=0.419783
epoch  19 | recon_loss=0.418983
epoch  20 | recon_loss=0.418373
epoch  21 | recon_loss=0.417992
epoch  22 | recon_loss=0.417828
epoch  23 | recon_loss=0.417501
epoch  24 | recon_loss=0.417374
epoch  25 | recon_loss=0.417133
epoch  26 | recon_loss=0.417036
epoch  27 | recon_loss=0.416860
epoch  28 | recon_loss=0.416732
epoch  29 | recon_loss=0.416614


In [4]:
from src.phaseI.inference import encode_snapshot
snapshot, points = next(realtime)

engine_id = 7

latent = encode_snapshot(
    model=model,
    snapshot=snapshot[engine_id],
    stats=stats,
    window_len=30,
    bin_stride=10,
)

print("observed cycles:", points[engine_id])
print("latent shape:", latent.shape)

observed cycles: 60
latent shape: (4, 16)


In [5]:
gru_model, history = run_phase2(
    latent_data=latent_data,
    train_metadata=data["train_metadata"],
    latent_dim=16,
)


epoch   0 | loss=0.7533 (hazard=0.7523, mono=0.0098) | val C-index=0.7059 *
epoch   1 | loss=0.6977 (hazard=0.6966, mono=0.0103) | val C-index=0.9412 *
epoch   2 | loss=0.6404 (hazard=0.6393, mono=0.0109) | val C-index=0.9412
epoch   3 | loss=0.5934 (hazard=0.5922, mono=0.0123) | val C-index=0.8824
epoch   4 | loss=0.5398 (hazard=0.5385, mono=0.0135) | val C-index=0.7059
epoch   5 | loss=0.5017 (hazard=0.5002, mono=0.0143) | val C-index=0.5882
epoch   6 | loss=0.4594 (hazard=0.4579, mono=0.0155) | val C-index=0.5882
epoch   7 | loss=0.4478 (hazard=0.4462, mono=0.0159) | val C-index=0.5882
epoch   8 | loss=0.4432 (hazard=0.4415, mono=0.0166) | val C-index=0.5882
epoch   9 | loss=0.3948 (hazard=0.3930, mono=0.0182) | val C-index=0.5882
epoch  10 | loss=0.3788 (hazard=0.3770, mono=0.0184) | val C-index=0.5882
epoch  11 | loss=0.3757 (hazard=0.3738, mono=0.0190) | val C-index=0.5882
epoch  12 | loss=0.3603 (hazard=0.3584, mono=0.0192) | val C-index=0.5882
epoch  13 | loss=0.3202 (hazard=0.

In [6]:
import torch
gru_model.eval()

with torch.no_grad():
    # latent: (T, 16)
    x = torch.from_numpy(latent[None]).float()

    lengths = torch.tensor([latent.shape[0]])

    hazard = gru_model(x, lengths)

In [13]:
hazard

tensor([[0.4901, 0.4570, 0.4366, 0.4336]])

In [16]:
for step in range(20):

    snapshot, points = next(realtime)

    latent = encode_snapshot(
        model=model,
        snapshot=snapshot[engine_id],
        stats=stats,
        window_len=30,
        bin_stride=10,
    )

    x = torch.from_numpy(latent[None]).float()
    lengths = torch.tensor([latent.shape[0]])

    with torch.no_grad():
        hazard = gru_model(x, lengths)

    print(
        f"observed={points[engine_id]:3d} | "
        f"bins={latent.shape[0]:2d} | "
        f"hazard={hazard.numpy()[0]}"
    )

observed= 68 | bins= 4 | hazard=[0.4900743  0.45702878 0.43662253 0.4335535 ]
observed= 69 | bins= 4 | hazard=[0.4900743  0.45702878 0.43662253 0.4335535 ]
observed= 70 | bins= 5 | hazard=[0.4900743  0.45702878 0.43662253 0.4335535  0.45084384]
observed= 71 | bins= 5 | hazard=[0.4900743  0.45702878 0.43662253 0.4335535  0.45084384]
observed= 72 | bins= 5 | hazard=[0.4900743  0.45702878 0.43662253 0.4335535  0.45084384]
observed= 73 | bins= 5 | hazard=[0.4900743  0.45702878 0.43662253 0.4335535  0.45084384]
observed= 73 | bins= 5 | hazard=[0.4900743  0.45702878 0.43662253 0.4335535  0.45084384]
observed= 74 | bins= 5 | hazard=[0.4900743  0.45702878 0.43662253 0.4335535  0.45084384]
observed= 75 | bins= 5 | hazard=[0.4900743  0.45702878 0.43662253 0.4335535  0.45084384]
observed= 75 | bins= 5 | hazard=[0.4900743  0.45702878 0.43662253 0.4335535  0.45084384]
observed= 76 | bins= 5 | hazard=[0.4900743  0.45702878 0.43662253 0.4335535  0.45084384]
observed= 77 | bins= 5 | hazard=[0.4900743 

In [7]:
print("Development:", len(data["train_ids"]))
print("Test:", len(data["test_ids"]))

extended = [
    eid for eid in data["train_ids"]
    if data["train_metadata"][eid]["is_extended"]
]

print("Extended:", len(extended))
print("Extended ratio:", len(extended) / len(data["train_ids"]))

Development: 70
Test: 30
Extended: 19
Extended ratio: 0.2714285714285714


In [8]:
print("Number of engines:", len(latent_data))

for engine_id in list(latent_data)[:5]:
    print(
        engine_id,
        latent_data[engine_id].shape
    )

Number of engines: 70
1 (11, 16)
2 (10, 16)
3 (10, 16)
4 (12, 16)
5 (15, 16)


In [9]:
train_metadata=data["train_metadata"]

In [10]:
from src.phaseII.core.prepare_data import prepare_training_data

train_records, val_records = prepare_training_data(
    latent_data=latent_data,
    train_metadata=data["train_metadata"],
    train_ratio=0.8,
    random_seed=42,
)

for name, records in [("train", train_records), ("val", val_records)]:
    print(f"\n{name}")

    for r in records:
        print(
            f"engine={r.engine_id:>3} | "
            f"bins={r.n_bins:>2} | "
            f"event={r.event_bin}"
        )


train
engine= 17 | bins=11 | event=10
engine= 10 | bins=11 | event=None
engine= 33 | bins=11 | event=None
engine= 93 | bins=10 | event=None
engine= 94 | bins=10 | event=None
engine= 43 | bins=10 | event=None
engine= 95 | bins=11 | event=None
engine=100 | bins=10 | event=None
engine= 76 | bins=10 | event=None
engine= 69 | bins=11 | event=None
engine= 77 | bins=11 | event=None
engine= 70 | bins= 9 | event=None
engine= 79 | bins=13 | event=12
engine= 62 | bins=10 | event=None
engine= 60 | bins=11 | event=None
engine= 47 | bins=11 | event=None
engine= 32 | bins=14 | event=13
engine= 84 | bins=10 | event=None
engine= 52 | bins=14 | event=13
engine= 82 | bins=10 | event=None
engine= 85 | bins=12 | event=11
engine= 24 | bins=10 | event=None
engine= 46 | bins=11 | event=None
engine= 63 | bins=10 | event=None
engine=  6 | bins=10 | event=None
engine= 80 | bins=10 | event=None
engine= 56 | bins=10 | event=None
engine= 49 | bins=10 | event=None
engine= 40 | bins=11 | event=None
engine= 16 | bins

In [11]:
import numpy as np
for name, records in [("train", train_records), ("val", val_records)]:

    normal_bins = [
        r.n_bins for r in records
        if r.event_bin is None
    ]

    extended_bins = [
        r.n_bins for r in records
        if r.event_bin is not None
    ]

    print(f"\n{name}")
    print("normal  :", normal_bins)
    print("extended:", extended_bins)

    print(
        "normal mean  =",
        np.mean(normal_bins) if normal_bins else None
    )

    print(
        "extended mean =",
        np.mean(extended_bins) if extended_bins else None
    )


train
normal  : [11, 11, 10, 10, 10, 11, 10, 10, 11, 11, 9, 10, 11, 11, 10, 10, 10, 11, 10, 10, 10, 10, 10, 11, 11, 10, 10, 11, 11, 10, 10, 9, 10, 11, 10, 10, 10, 10, 11, 10]
extended: [11, 13, 14, 14, 12, 15, 13, 13, 12, 12, 15, 8, 15, 11, 11]
normal mean  = 10.3
extended mean = 12.6

val
normal  : [11, 11, 10, 11, 10, 10, 10, 10, 11, 11, 11]
extended: [13, 10, 14, 15]
normal mean  = 10.545454545454545
extended mean = 13.0


In [12]:
for engine_id in data["train_ids"]:
    if data["train_metadata"][engine_id]["is_extended"]:
        print(
            engine_id,
            "cycles observed =",
            len(data["train_data"][engine_id]),
            "latent bins =",
            latent_data[engine_id].shape[0],
        )

4 cycles observed = 142 latent bins = 12
5 cycles observed = 176 latent bins = 15
16 cycles observed = 171 latent bins = 15
17 cycles observed = 136 latent bins = 11
22 cycles observed = 155 latent bins = 13
28 cycles observed = 135 latent bins = 11
32 cycles observed = 169 latent bins = 14
38 cycles observed = 153 latent bins = 13
39 cycles observed = 108 latent bins = 8
41 cycles observed = 167 latent bins = 14
44 cycles observed = 154 latent bins = 13
52 cycles observed = 163 latent bins = 14
58 cycles observed = 127 latent bins = 10
64 cycles observed = 138 latent bins = 11
79 cycles observed = 152 latent bins = 13
81 cycles observed = 175 latent bins = 15
85 cycles observed = 143 latent bins = 12
92 cycles observed = 142 latent bins = 12
97 cycles observed = 178 latent bins = 15
